# Vision-Driven Computer Use

`06_BrowserAgent_Computer_Use_Applied.ipynb` teaches the shape of a computer-use loop, but
is explicit that it is a simulation: its `screenshot_text()` returns a *text description* of
the page. The agent never sees anything.

This notebook closes that. The agent receives an actual **PNG image** of a UI, and has to
work out from pixels where things are and what to click. Nothing is described to it in
words.

That change is not cosmetic. When the observation is text, someone has already done the
hard part — parsing the interface into named elements. When it is pixels, the model has to
do that itself, and everything that makes computer use unreliable shows up.

## Learning objectives

1. Build an act–observe loop whose observation is an image, not a description.
2. Send a screenshot to a vision model and parse a structured action back.
3. Watch raw-coordinate clicking fail, then fix it with a coordinate ruler drawn on the
   screenshot — and measure the difference in misses.
4. Use it as a **frontend testing** agent: give it a goal and let it find the bug.
5. Name why coordinate-based control is brittle, and what production systems do instead.

## Where this fits

- `06_BrowserAgent_Computer_Use_Applied.ipynb` — the same loop with text observations.
  Read it first; this is the pixel version.
- `07_Hosted_vs_Client_Side_Tools.ipynb` — OpenAI's `computer_use_preview` is the hosted
  version of everything below, with the trade-offs that notebook describes.

## Dependencies, deliberately none new

The UI is rendered with **Pillow**, which is already installed, and the screenshots are real
PNGs. No Playwright, no Selenium, no headless Chromium download — none of which are
dependencies of this repo.

What that costs: the GUI is drawn rather than browsed. What it keeps: real pixels, real
vision, real coordinates, and a loop that behaves the way a browser-driven one does. The
final section explains exactly what changes when you swap the renderer for a real browser.

## Prerequisites

`OPENAI_API_KEY` in the project-root `.env`. Vision calls cost more than text — roughly
`steps × screenshots`, a few cents.

In [ ]:
# ============ SETUP ============
import base64
import io
import json

from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont

load_dotenv()
client = OpenAI()

VISION_MODEL = "gpt-4o-mini"   # must accept image input
# The goal needs 4 successful clicks (+, +, coupon, place) plus a `done`, so a cap of 6
# would decide the outcome rather than bound it. 10 leaves room to recover from misses
# while still stopping a confused agent.
MAX_STEPS = 10

def _font(size: int):
    for path in ("/System/Library/Fonts/Supplemental/Arial.ttf",
                 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                 "C:/Windows/Fonts/arial.ttf"):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()

## 1. A UI made of pixels

`Screen` draws a small checkout page and tracks its own state. The important detail is what
it does **not** expose: there is no `get_elements()`, no accessibility tree, no DOM. The only
way out is `screenshot()`, which returns a PNG.

Clicks arrive as `(x, y)`. The screen decides what, if anything, was hit — exactly as a real
GUI does.

In [ ]:
# ============ THE UI ============
class Screen:
    """A drawn checkout page. Observable only as pixels; clickable only by coordinate."""

    W, H = 520, 360
    GRID = 40          # ruler spacing, in logical units

    def __init__(self, coupon_button_broken: bool = False):
        self.qty = 1
        self.coupon_applied = False
        self.placed = False
        self.coupon_button_broken = coupon_button_broken   # the bug, for section 5
        self.click_log: list[tuple[int, int, str]] = []
        # name -> (x1, y1, x2, y2). The agent never sees this.
        self.boxes = {
            "qty_plus":   (250, 96, 286, 130),
            "qty_minus":  (196, 96, 232, 130),
            "coupon":     (40, 190, 210, 228),
            "place":      (40, 262, 260, 306),
        }

    def _hit(self, x: int, y: int) -> str | None:
        for name, (x1, y1, x2, y2) in self.boxes.items():
            if x1 <= x <= x2 and y1 <= y <= y2:
                return name
        return None

    def click(self, x: int, y: int) -> str:
        """Coordinates are always LOGICAL (0..W, 0..H), whatever the screenshot was scaled to."""
        target = self._hit(x, y)
        self.click_log.append((x, y, target or "MISS"))
        if target == "qty_plus":
            self.qty += 1
        elif target == "qty_minus":
            self.qty = max(1, self.qty - 1)
        elif target == "coupon":
            if not self.coupon_button_broken:
                self.coupon_applied = True
        elif target == "place":
            self.placed = True
        return target or "MISS"

    def _draw(self) -> Image.Image:
        img = Image.new("RGB", (self.W, self.H), "#f4f4f6")
        d = ImageDraw.Draw(img)
        big, mid, small = _font(20), _font(16), _font(13)

        d.text((40, 30), "Checkout", fill="#111", font=big)
        d.text((40, 70), "Wireless Mouse", fill="#333", font=mid)

        # quantity stepper
        d.rectangle(self.boxes["qty_minus"], outline="#555", width=2, fill="white")
        d.text((208, 103), "-", fill="#111", font=mid)
        d.text((240, 103), str(self.qty), fill="#111", font=mid)
        d.rectangle(self.boxes["qty_plus"], outline="#555", width=2, fill="white")
        d.text((262, 103), "+", fill="#111", font=mid)

        # coupon
        fill = "#cdebd6" if self.coupon_applied else "white"
        d.rectangle(self.boxes["coupon"], outline="#555", width=2, fill=fill)
        label = "Coupon applied" if self.coupon_applied else "Apply coupon"
        d.text((54, 202), label, fill="#111", font=mid)

        # place order
        d.rectangle(self.boxes["place"], outline="#1a5", width=3,
                    fill="#1a5" if not self.placed else "#888")
        d.text((60, 277), "Order placed" if self.placed else "Place order",
               fill="white", font=mid)

        d.text((40, 326), f"total: ${12.50 * self.qty:.2f}", fill="#333", font=small)
        return img

    def render(self, grid: bool = False, scale: int = 1) -> Image.Image:
        """`grid` overlays a labelled coordinate ruler; `scale` enlarges the whole image.

        The ruler is labelled in LOGICAL units, so a coordinate read off it is directly
        clickable no matter what `scale` is. Note what the ruler does NOT do: it never
        names or outlines a control. It supplies a reference frame, not an answer key.
        """
        img = self._draw()
        if grid:
            d = ImageDraw.Draw(img)
            f = _font(11)
            for x in range(0, self.W, self.GRID):
                d.line([(x, 0), (x, self.H)], fill="#c9c9d4", width=1)
                d.text((x + 2, 1), str(x), fill="#d0021b", font=f)
            for y in range(self.GRID, self.H, self.GRID):
                d.line([(0, y), (self.W, y)], fill="#c9c9d4", width=1)
                d.text((2, y + 1), str(y), fill="#d0021b", font=f)
        if scale != 1:
            img = img.resize((self.W * scale, self.H * scale), Image.LANCZOS)
        return img

    # kept so `screen.screenshot()` still reads naturally in the plain loop
    screenshot = render


def to_data_url(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


demo = Screen()
print(f"  plain:  {demo.render().size[0]}x{demo.render().size[1]} PNG")
_ruled = demo.render(grid=True, scale=2)
print(f"  ruled:  {_ruled.size[0]}x{_ruled.size[1]} PNG")
demo.render()

## 2. The agent sees pixels

One call: the screenshot goes in as an image, a JSON action comes back. The prompt gives the
model the image dimensions and the action vocabulary — and nothing about what is on screen.

Note the action space is deliberately tiny: `click(x, y)` or `done`. Every extra verb is
another thing the model can get wrong.

In [ ]:
# ============ ONE VISION STEP ============
_CONTRACT = (
    "Reply ONLY with JSON, no prose, no code fences:\n"
    '  {"action": "click", "x": <int>, "y": <int>, "why": "<short>"}\n'
    '  {"action": "done", "why": "<short>"}\n'
    "Click the CENTRE of the control you intend to press."
)

SYSTEM = (
    "You operate a GUI by looking at screenshots. You are given a PNG of the current screen "
    f"({Screen.W}x{Screen.H} pixels, origin top-left). Decide the single next action.\n"
    + _CONTRACT
)

SYSTEM_RULER = (
    "You operate a GUI by looking at screenshots. A coordinate ruler is drawn over the "
    f"screenshot: red numbers along the top edge are x, red numbers down the left edge are "
    f"y, and gridlines fall every {Screen.GRID} units.\n"
    f"Read the target's position off those rulers and report it in RULER units "
    f"(x from 0 to {Screen.W}, y from 0 to {Screen.H}) — not in raw image pixels.\n"
    "State which gridlines the control sits between before you commit to a number.\n"
    + _CONTRACT
)


def decide(img: Image.Image, goal: str, history: list[str], system: str = SYSTEM) -> dict:
    past = ("\n".join(f"- {h}" for h in history)) or "- (nothing yet)"
    reply = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": [
                {"type": "text",
                 "text": f"Goal: {goal}\n\nActions so far:\n{past}\n\nNext action?"},
                {"type": "image_url", "image_url": {"url": to_data_url(img)}},
            ]},
        ],
    )
    raw = (reply.choices[0].message.content or "").strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"action": "done", "why": f"unparseable reply: {raw[:60]}"}

## 3. The loop

Screenshot → decide → act → screenshot. The agent is told only whether its click **hit
something or missed**, never what the thing was called. Missing is information too, and a
capable agent should correct after one.

In [ ]:
# ============ ACT-OBSERVE LOOP ============
GOAL = "Set the quantity to 3, apply the coupon, then place the order."


def run(goal: str, screen: Screen, max_steps: int = MAX_STEPS, verbose: bool = True,
        grid: bool = False, scale: int = 1):
    system = SYSTEM_RULER if grid else SYSTEM
    history: list[str] = []
    for step in range(1, max_steps + 1):
        action = decide(screen.render(grid=grid, scale=scale), goal, history, system)
        if action.get("action") == "done":
            if verbose:
                print(f"  {step}. done — {action.get('why','')}")
            break
        x, y = int(action.get("x", -1)), int(action.get("y", -1))
        hit = screen.click(x, y)
        note = f"clicked ({x},{y}) -> {hit}"
        history.append(note)
        if verbose:
            print(f"  {step}. {note}   [{action.get('why','')[:44]}]")
    return screen


def report(screen: Screen, label: str) -> int:
    misses = sum(1 for *_, t in screen.click_log if t == "MISS")
    print(f"\n  [{label}] qty={screen.qty}  coupon={screen.coupon_applied}  "
          f"placed={screen.placed}")
    print(f"  [{label}] clicks: {len(screen.click_log)} ({misses} missed)")
    return misses


print("GOAL: set quantity to 3, apply the coupon, then place the order")
print("Observation: a raw 520x360 screenshot, no coordinate hints.\n")
plain = run(GOAL, Screen())
plain_misses = report(plain, "raw pixels")

### Discussion of the output

Watch the `MISS` count. That number is the entire reliability story of coordinate-based
computer use, and it is why this is the least dependable tool-use pattern in the repo.

A typical run misses **every single click**. Read the trace carefully, because the shape of
the failure matters more than the count:

```
1. clicked (269,184) -> MISS   [To increase the quantity to 3]
2. clicked (282,191) -> MISS   [To increase the quantity to 3.]
3. clicked (294,191) -> MISS   [To increase the quantity to 3.]
4. clicked (322,191) -> MISS   [To increase the quantity to 3.]
```

Three things are visible there:

1. **The intent is correct throughout.** It knows it wants the `+` control. Recognising
   *what* is on screen and knowing *where* it is are different abilities, and vision models
   are markedly better at the first.
2. **The first click was already half-right.** `x=269` is inside the `+` button's box
   (250–286). Only `y` was wrong — by about 60px, which put it in the coupon row.
3. **Feeding back `MISS` did not help.** This loop is generous: `run()` appends
   `clicked (x,y) -> MISS` to the history and hands it back on the next call. The agent was
   told it missed, six times, and never revised `y` once. It held the wrong row fixed and
   searched only the axis that was already right.

That last point is the one to take away. A miss is not a crash, and here it is not even
silent — and it *still* does not get corrected. In a real harness you get the next
screenshot and nothing else, so the same failure is genuinely invisible: **not an error, a
no-op loop.** The step cap is what turns it into a bounded failure instead of an unbounded
bill.

So the raw-coordinate loop does not work with this model. The next section asks whether that
is a property of the task or a property of how we posed it.

## 4. The same loop, with a coordinate ruler

Nothing about the agent changes. The screen, the model, the action vocabulary, the loop —
all identical. Two things change about the **observation**:

- The screenshot is rendered at **2×**, so each control covers four times the image tokens.
  A 36×34px button on a 520×360 canvas is roughly 0.6% of the image; after the provider's
  resize it is a handful of tokens wide.
- A **coordinate ruler** is drawn over it: gridlines every 40 units, labelled along the top
  and left edges.

Be clear about what the ruler is and is not. It does not name a control, outline one, or
number the clickable regions — the agent still has to find the `+` button itself. It
supplies a *reference frame*: instead of estimating "about 270 across, maybe 180 down" from
nothing, the model can read that the button sits between the `240` and `280` gridlines.

This is a stripped-down version of **Set-of-Mark prompting**, the standard mitigation for
weak visual grounding. Because the ruler is labelled in logical units, a coordinate read off
it is directly clickable regardless of the 2× scaling — the `click()` API never changes.

In [ ]:
# ============ SAME LOOP, RULED SCREENSHOT ============
print("Same goal, same model, same loop.")
print("Observation: 1040x720 screenshot with a labelled coordinate grid.\n")
ruled = run(GOAL, Screen(), grid=True, scale=2)
ruled_misses = report(ruled, "ruled")

print("\n  --- comparison ---")
print(f"  raw pixels : {len(plain.click_log)} clicks, {plain_misses} missed")
print(f"  ruled 2x   : {len(ruled.click_log)} clicks, {ruled_misses} missed")

# what the model was actually looking at
Screen().render(grid=True, scale=2)

### What the comparison shows

The agent did not get smarter. The observation got easier to measure against, and the miss
rate moves accordingly.

This is the practical lesson underneath the whole notebook, and it generalises past
screenshots: **when a model is bad at producing a continuous quantity, give it something
discrete to read the quantity off.** Gridlines here; numbered element overlays in
Set-of-Mark; a DOM selector in a real browser. In each case you are replacing an estimate
with a lookup.

Two caveats worth holding onto:

- **The ruler costs image tokens and clutters the screen.** On a dense real page, gridlines
  over content hurt legibility, which is why production Set-of-Mark systems label *detected
  elements* rather than drawing a grid over everything.
- **It does not make coordinates reliable, only less unreliable.** If you have a selector
  or an accessibility node, the ruler is still the worse option. Section 6.

## 5. As a frontend testing agent

This is OpenAI's own example use case for computer use, and it is a better fit than
general automation, because a test has something the open-ended case lacks: **a
verifiable end state.**

Below, the coupon button is wired to do nothing — a plausible frontend regression. The
agent is not told. We give it the same goal and then assert on the result. It runs with the
ruled observation from section 4, because a test that cannot reach the control under test
tells you nothing.

In [ ]:
# ============ THE AGENT MEETS A BROKEN BUTTON ============
broken = Screen(coupon_button_broken=True)
print("GOAL (against a build where the coupon button is broken)\n")
run(GOAL, broken, grid=True, scale=2)

print("\n  --- assertions ---")
checks = {
    "quantity is 3":   broken.qty == 3,
    "coupon applied":  broken.coupon_applied,
    "order placed":    broken.placed,
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

# Both branches matter. A failing assertion on its own cannot tell you which one you are in.
coupon_clicks = [c for c in broken.click_log if c[2] == "coupon"]
print()
if broken.coupon_applied:
    print("  diagnosis: coupon applied -> the control works.")
elif coupon_clicks:
    print(f"  diagnosis: the agent hit the coupon control {len(coupon_clicks)}x and state "
          f"never changed -> the control is broken, not the agent.")
else:
    print(f"  diagnosis: the agent never reached the coupon control in "
          f"{len(broken.click_log)} clicks -> agent failure, not a UI verdict.")
    print("             This run says NOTHING about whether the button works. "
          "Treat it as an errored test, not a failing one.")

### Why that diagnosis matters

A failing assertion alone is nearly useless: *"coupon not applied"* could mean the button is
broken, or that the agent never found it. Those need different people to fix them.

The click log separates them, and the code above prints all three verdicts rather than only
the interesting one:

| Click log | Verdict | Whose problem |
|---|---|---|
| Hit the control, state changed | pass | nobody |
| Hit the control, state unchanged | **fail** — the UI is broken | frontend |
| Never hit the control | **errored** — the UI is unverified | whoever owns the agent |

The third row is the one teams get wrong. It is not a failing test; it is a test that did
not run. Reporting it as a failure sends someone hunting a frontend bug that may not exist —
and reporting it as a pass is worse. Any computer-use test worth running needs that
three-way distinction built in.

## 6. What changes with a real browser

Everything above is a genuine vision loop — real PNGs, real pixel coordinates, real
misclicks. What a real browser would change:

| | Here | Playwright / Selenium |
|---|---|---|
| Rendering | Pillow draws it | A real engine, real fonts, real layout |
| What can break | Only what I coded | Scroll position, overlays, timing, focus, animation |
| Element access | Coordinates only | Coordinates **and** selectors / accessibility tree |
| Setup | none | `pip install playwright && playwright install chromium` (~300 MB) |

The loop shape does not change, which is the point of learning it here. What changes is the
*failure* surface: real pages move under you, and a screenshot can be stale by the time the
click lands. The ruler trick from section 4 survives the move, but it is strictly a
fallback — a real page gives you something better.

**And the practical conclusion:** if a selector or an accessibility tree is available, use
it. Coordinate clicking is the fallback for when nothing else is exposed, not the default.
Production computer-use systems reach for pixels last, not first — hence the hosted
`computer_use_preview` tool in `07_Hosted_vs_Client_Side_Tools.ipynb`, which exists precisely
because doing this well is harder than it looks.

## Key takeaways

1. **Text observations hide the hard part.** Once the screen is pixels, the agent must
   localise controls itself, and that is where computer use actually fails.
2. **Recognising and locating are different skills.** Expect an accurate description of a
   control paired with coordinates tens of pixels off — often wrong on one axis only.
3. **Telling the agent it missed may not be enough.** This loop feeds `-> MISS` straight
   back, and a weak model still re-clicks the same row repeatedly. Error feedback is not
   error correction.
4. **The signature failure is a no-op**, not an exception. A missed click changes nothing,
   so a step cap is not optional — it is the only bound on the loop. Size it above what a
   perfect run needs, or the cap decides the outcome instead of bounding it.
5. **Give the model something discrete to read.** A labelled ruler on the screenshot turns
   an estimate into a lookup, and the miss count moves without the agent changing at all.
   That is Set-of-Mark prompting in miniature.
6. **Frontend testing is the strongest use case**, because assertions give the loop a
   verifiable end state that open-ended automation lacks.
7. **Log the clicks and what they hit**, and report three outcomes, not two: passed, failed,
   and *never reached the control*. The third is an errored test, not a failing one.
8. **Prefer selectors to coordinates.** Pixels are the fallback for interfaces that expose
   nothing else.

### Next

- `07_Hosted_vs_Client_Side_Tools.ipynb` — `computer_use_preview` runs this loop on the
  provider's side, with the control trade-offs set out there.